# SQL Complete Practice — Part 10: Database Relationships
**Topics 152–161 | MySQL | Ankanhaldar**

This notebook explains how tables relate to one another in a relational database, how foreign keys represent those relationships, and how to design common relationship patterns.

> **Execution note:** These are MySQL SQL cells, not Python. Run them in MySQL Workbench, VS Code with SQLTools, or a compatible SQL notebook/kernel. A standard Python kernel does not execute SQL directly.

> **Practice database:** Run the setup cell first. The examples use separate table names for each relationship pattern so that each example can be practised independently.

In [ ]:
CREATE DATABASE IF NOT EXISTS ankan_sql_practice;
USE ankan_sql_practice;

-- Parent table used by several relationship examples
CREATE TABLE IF NOT EXISTS relationship_demo_department (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(80) NOT NULL UNIQUE
);

INSERT INTO relationship_demo_department (department_id, department_name)
VALUES
    (10, 'IT'),
    (20, 'HR'),
    (30, 'Finance')
ON DUPLICATE KEY UPDATE department_name = VALUES(department_name);

SELECT * FROM relationship_demo_department;

**Setup note:** The examples use `CREATE TABLE IF NOT EXISTS` and fixed sample IDs. If you change a table definition while practising, use a new table name or carefully reset only your disposable practice tables.

## Topic 152: What Is a Database Relationship?

A **database relationship** describes how records in one table are associated with records in another table.

Relational databases use relationships to:
- Organize data into related tables.
- Reduce unnecessary duplication.
- Maintain consistency using keys and constraints.
- Retrieve connected information using joins.

A common relationship uses a primary key in one table and a foreign key in another table. The foreign key stores the referenced key value.

### SQL Example — Topic 152

In [ ]:
CREATE TABLE relationship_demo_employee (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    department_id INT,
    CONSTRAINT fk_rel_employee_department
        FOREIGN KEY (department_id)
        REFERENCES relationship_demo_department(department_id)
);

INSERT INTO relationship_demo_employee
    (employee_id, employee_name, department_id)
VALUES
    (1, 'Amit', 10),
    (2, 'Riya', 20),
    (3, 'Suman', 10);

SELECT
    e.employee_name,
    d.department_name
FROM relationship_demo_employee AS e
JOIN relationship_demo_department AS d
    ON e.department_id = d.department_id;

**Expected result / explanation**

The employee table stores a department ID rather than repeating the department name in every employee row. The foreign key connects the employee record to a valid department.

## Topic 153: One-to-One Relationship (1:1)

In a **one-to-one relationship**, one record in table A is associated with at most one record in table B, and vice versa.

A common implementation is to place a foreign key in the second table and add a `UNIQUE` constraint to it. The unique constraint prevents multiple rows from referring to the same parent.

Use this pattern when separating optional or sensitive details from a main entity.

### SQL Example — Topic 153

In [ ]:
CREATE TABLE relationship_demo_person (
    person_id INT PRIMARY KEY,
    person_name VARCHAR(80) NOT NULL
);

CREATE TABLE relationship_demo_person_passport (
    passport_id INT PRIMARY KEY,
    person_id INT NOT NULL UNIQUE,
    passport_number VARCHAR(30) NOT NULL UNIQUE,
    CONSTRAINT fk_rel_passport_person
        FOREIGN KEY (person_id)
        REFERENCES relationship_demo_person(person_id)
);

INSERT INTO relationship_demo_person
    (person_id, person_name)
VALUES
    (1, 'Asha'),
    (2, 'Dev');

INSERT INTO relationship_demo_person_passport
    (passport_id, person_id, passport_number)
VALUES
    (101, 1, 'P-10001'),
    (102, 2, 'P-10002');

SELECT
    p.person_name,
    pp.passport_number
FROM relationship_demo_person AS p
JOIN relationship_demo_person_passport AS pp
    ON p.person_id = pp.person_id;

**Expected result / explanation**

`person_id` in the passport table is both a foreign key and `UNIQUE`, so one person can have at most one passport row in this example. A person can still have no passport row because the relationship is optional from the person's side.

## Topic 154: One-to-Many Relationship (1:N)

In a **one-to-many relationship**, one parent record can be associated with multiple child records, while each child record is associated with at most one parent.

This is commonly implemented by placing a foreign key in the child table.

Examples:
- One department has many employees.
- One customer places many orders.
- One author writes many books.

### SQL Example — Topic 154

In [ ]:
CREATE TABLE relationship_demo_customer (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(80) NOT NULL
);

CREATE TABLE relationship_demo_customer_order (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_amount DECIMAL(10, 2) NOT NULL,
    CONSTRAINT fk_rel_order_customer
        FOREIGN KEY (customer_id)
        REFERENCES relationship_demo_customer(customer_id)
);

INSERT INTO relationship_demo_customer
    (customer_id, customer_name)
VALUES
    (1, 'Mina'),
    (2, 'Kabir');

INSERT INTO relationship_demo_customer_order
    (order_id, customer_id, order_amount)
VALUES
    (1001, 1, 1500.00),
    (1002, 1, 2200.00),
    (1003, 2, 900.00);

SELECT
    c.customer_name,
    o.order_id,
    o.order_amount
FROM relationship_demo_customer AS c
JOIN relationship_demo_customer_order AS o
    ON c.customer_id = o.customer_id;

**Expected result / explanation**

Customer Mina has two orders, while Kabir has one. The foreign key is stored in the many-side table, `relationship_demo_customer_order`.

## Topic 155: Many-to-Many Relationship (M:N)

In a **many-to-many relationship**, multiple records in table A can relate to multiple records in table B.

A relational database normally implements this using a **junction table** (also called a bridge or associative table). The junction table stores a foreign key to each participating table.

Example: a student can enrol in many courses, and each course can have many students.

### SQL Example — Topic 155

In [ ]:
CREATE TABLE relationship_demo_student (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(80) NOT NULL
);

CREATE TABLE relationship_demo_course (
    course_id INT PRIMARY KEY,
    course_name VARCHAR(100) NOT NULL
);

CREATE TABLE relationship_demo_student_course (
    student_id INT NOT NULL,
    course_id INT NOT NULL,
    enrollment_date DATE NOT NULL,
    PRIMARY KEY (student_id, course_id),
    CONSTRAINT fk_rel_sc_student
        FOREIGN KEY (student_id)
        REFERENCES relationship_demo_student(student_id),
    CONSTRAINT fk_rel_sc_course
        FOREIGN KEY (course_id)
        REFERENCES relationship_demo_course(course_id)
);

INSERT INTO relationship_demo_student
    (student_id, student_name)
VALUES
    (1, 'Amit'),
    (2, 'Riya');

INSERT INTO relationship_demo_course
    (course_id, course_name)
VALUES
    (101, 'Database Systems'),
    (102, 'Python Programming');

INSERT INTO relationship_demo_student_course
    (student_id, course_id, enrollment_date)
VALUES
    (1, 101, '2026-01-10'),
    (1, 102, '2026-01-11'),
    (2, 101, '2026-01-12');

SELECT
    s.student_name,
    c.course_name,
    sc.enrollment_date
FROM relationship_demo_student_course AS sc
JOIN relationship_demo_student AS s
    ON sc.student_id = s.student_id
JOIN relationship_demo_course AS c
    ON sc.course_id = c.course_id;

**Expected result / explanation**

The junction table allows one student to have multiple courses and one course to have multiple students. Its composite primary key prevents the same student-course pair from being entered twice.

## Topic 156: Self-Referencing Relationship

A **self-referencing relationship** occurs when a table has a foreign key that references its own primary key.

It is useful for hierarchical or recursive structures, such as:
- Employee and manager.
- Category and parent category.
- Organization reporting structure.

A nullable foreign key can represent a top-level record that has no parent.

### SQL Example — Topic 156

In [ ]:
CREATE TABLE relationship_demo_org_employee (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    manager_id INT NULL,
    CONSTRAINT fk_rel_org_manager
        FOREIGN KEY (manager_id)
        REFERENCES relationship_demo_org_employee(employee_id)
);

INSERT INTO relationship_demo_org_employee
    (employee_id, employee_name, manager_id)
VALUES
    (1, 'Director', NULL),
    (2, 'Team Lead', 1),
    (3, 'Developer', 2),
    (4, 'Tester', 2);

SELECT
    e.employee_name AS employee,
    m.employee_name AS manager
FROM relationship_demo_org_employee AS e
LEFT JOIN relationship_demo_org_employee AS m
    ON e.manager_id = m.employee_id
ORDER BY e.employee_id;

**Expected result / explanation**

The `manager_id` column references another row in the same table. The director has no manager, so `manager_id` is `NULL`.

## Topic 157: Parent and Child Tables

A **parent table** contains the referenced key. A **child table** contains the foreign key that points to the parent.

Important points:
- Insert the parent row before inserting a child row that references it.
- A child foreign key value must match an existing referenced key, unless the foreign key is nullable and the value is `NULL`.
- A parent row cannot normally be deleted while referenced child rows exist, unless a suitable referential action is configured.
- A foreign key helps enforce referential integrity.

### SQL Example — Topic 157

In [ ]:
CREATE TABLE relationship_demo_parent (
    parent_id INT PRIMARY KEY,
    parent_name VARCHAR(80) NOT NULL
);

CREATE TABLE relationship_demo_child (
    child_id INT PRIMARY KEY,
    parent_id INT NOT NULL,
    child_name VARCHAR(80) NOT NULL,
    CONSTRAINT fk_rel_child_parent
        FOREIGN KEY (parent_id)
        REFERENCES relationship_demo_parent(parent_id)
);

-- Insert parent first
INSERT INTO relationship_demo_parent (parent_id, parent_name)
VALUES (1, 'Parent One');

-- Then insert its child
INSERT INTO relationship_demo_child (child_id, parent_id, child_name)
VALUES (101, 1, 'Child One');

SELECT
    p.parent_name,
    c.child_name
FROM relationship_demo_parent AS p
JOIN relationship_demo_child AS c
    ON p.parent_id = c.parent_id;

**Expected result / explanation**

The parent row is inserted first, followed by the child row that references it. The join displays their relationship.

## Topic 158: Referential Integrity

**Referential integrity** means that relationships between tables remain valid.

A foreign key helps ensure that a child row references an existing parent key. It prevents orphaned references.

Common actions:
- `RESTRICT` / `NO ACTION` — prevent a parent change or deletion when referenced child rows exist (the exact timing can vary by engine).
- `CASCADE` — propagate a parent update or deletion to matching child rows.
- `SET NULL` — set the child foreign key to `NULL`, if the column allows it.
- `SET DEFAULT` — not supported as a foreign-key action by InnoDB in MySQL.

Choose actions according to the business rules. Cascading deletion should be used carefully.

### SQL Example — Topic 158

In [ ]:
CREATE TABLE relationship_demo_fk_parent (
    parent_id INT PRIMARY KEY,
    parent_name VARCHAR(80) NOT NULL
);

CREATE TABLE relationship_demo_fk_child (
    child_id INT PRIMARY KEY,
    parent_id INT NULL,
    child_name VARCHAR(80) NOT NULL,
    CONSTRAINT fk_rel_fk_child_parent
        FOREIGN KEY (parent_id)
        REFERENCES relationship_demo_fk_parent(parent_id)
        ON DELETE SET NULL
        ON UPDATE CASCADE
);

INSERT INTO relationship_demo_fk_parent (parent_id, parent_name)
VALUES (10, 'Parent A');

INSERT INTO relationship_demo_fk_child (child_id, parent_id, child_name)
VALUES (100, 10, 'Child A');

SELECT * FROM relationship_demo_fk_child;

-- Demonstration of ON DELETE SET NULL:
-- DELETE FROM relationship_demo_fk_parent WHERE parent_id = 10;
-- SELECT * FROM relationship_demo_fk_child;

**Expected result / explanation**

The child table uses `ON DELETE SET NULL`, so deleting its parent sets the child's `parent_id` to `NULL`. The demonstration DELETE is commented out to avoid changing data unless you intentionally choose to test it.

## Topic 159: ER Diagram (Entity-Relationship Diagram)

An **ER diagram** is a visual model of a database. It shows entities (usually tables), their attributes, and the relationships between them.

Common notation:
- Entity — rectangle.
- Attribute — listed inside or connected to an entity.
- Relationship — line connecting entities.
- Cardinality — indicates one-to-one, one-to-many, or many-to-many.

ER diagrams help plan a database before creating tables. A many-to-many relationship is usually converted into two one-to-many relationships through a junction table.

### SQL Example — Topic 159

In [ ]:
-- Example relational structure represented by an ER model:
-- DEPARTMENT (1) -------- (M) EMPLOYEE
-- STUDENT (1) ------------ (M) STUDENT_COURSE
-- COURSE   (1) ----------- (M) STUDENT_COURSE

CREATE TABLE relationship_demo_er_department (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(80) NOT NULL
);

CREATE TABLE relationship_demo_er_employee (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    department_id INT NOT NULL,
    FOREIGN KEY (department_id)
        REFERENCES relationship_demo_er_department(department_id)
);

**Expected result / explanation**

The SQL structure implements a one-to-many ER relationship: one department can be associated with many employees. The SQL comments show how a many-to-many student-course relationship is represented through a junction table.

## Topic 160: Cardinality and Optionality

**Cardinality** describes how many records can participate in a relationship.

Common cardinalities:
- One-to-one (1:1)
- One-to-many (1:N)
- Many-to-many (M:N)

**Optionality** describes whether participation is required:
- Mandatory: a related record must exist.
- Optional: a related record may be absent.

In SQL, `NOT NULL` on a foreign-key column generally requires each child row to provide a parent reference. A nullable foreign-key column allows a child row to have no parent reference. A `UNIQUE` foreign key can help enforce one-to-one cardinality.

### SQL Example — Topic 160

In [ ]:
CREATE TABLE relationship_demo_cardinality_parent (
    parent_id INT PRIMARY KEY,
    parent_name VARCHAR(80) NOT NULL
);

-- Mandatory many-to-one: parent_id is NOT NULL
CREATE TABLE relationship_demo_cardinality_mandatory_child (
    child_id INT PRIMARY KEY,
    parent_id INT NOT NULL,
    FOREIGN KEY (parent_id)
        REFERENCES relationship_demo_cardinality_parent(parent_id)
);

-- Optional many-to-one: parent_id allows NULL
CREATE TABLE relationship_demo_cardinality_optional_child (
    child_id INT PRIMARY KEY,
    parent_id INT NULL,
    FOREIGN KEY (parent_id)
        REFERENCES relationship_demo_cardinality_parent(parent_id)
);

-- One-to-one: UNIQUE prevents multiple child rows per parent
CREATE TABLE relationship_demo_cardinality_one_to_one (
    detail_id INT PRIMARY KEY,
    parent_id INT NOT NULL UNIQUE,
    FOREIGN KEY (parent_id)
        REFERENCES relationship_demo_cardinality_parent(parent_id)
);

**Expected result / explanation**

The mandatory child table requires a parent ID. The optional child table permits `NULL`. The one-to-one table adds `UNIQUE` to prevent multiple detail rows for the same parent.

## Topic 161: Mapping ER Relationships to Tables

Mapping converts an ER model into relational tables and constraints.

Common mapping rules:
- 1:1 — place a foreign key on one side and make it `UNIQUE`; choose the side based on optionality and design needs.
- 1:N — place the foreign key on the many side.
- M:N — create a junction table with foreign keys to both entities.
- Self-referencing — place a foreign key in the same table.
- Relationship attributes — store them in the junction table when they describe the association itself.

Choose primary keys and nullability based on the business rules.

### SQL Example — Topic 161

In [ ]:
-- 1:N mapping: one department to many employees
CREATE TABLE relationship_demo_mapping_department (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(80) NOT NULL
);

CREATE TABLE relationship_demo_mapping_employee (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    department_id INT NOT NULL,
    FOREIGN KEY (department_id)
        REFERENCES relationship_demo_mapping_department(department_id)
);

-- M:N mapping: students and courses through an enrollment table
CREATE TABLE relationship_demo_mapping_student (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(80) NOT NULL
);

CREATE TABLE relationship_demo_mapping_course (
    course_id INT PRIMARY KEY,
    course_name VARCHAR(100) NOT NULL
);

CREATE TABLE relationship_demo_mapping_enrollment (
    student_id INT NOT NULL,
    course_id INT NOT NULL,
    enrollment_date DATE NOT NULL,
    PRIMARY KEY (student_id, course_id),
    FOREIGN KEY (student_id)
        REFERENCES relationship_demo_mapping_student(student_id),
    FOREIGN KEY (course_id)
        REFERENCES relationship_demo_mapping_course(course_id)
);

**Expected result / explanation**

The first design places the department foreign key in the employee table (1:N). The second design uses an enrollment junction table with a composite primary key (M:N).

## Quick Revision Table

| Relationship | Meaning | Common implementation |
|---|---|---|
| One-to-one (1:1) | One row relates to at most one row | Foreign key plus `UNIQUE` |
| One-to-many (1:N) | One parent relates to many children | Foreign key on the many side |
| Many-to-many (M:N) | Many rows relate to many rows | Junction table with two foreign keys |
| Self-referencing | Rows relate to other rows in the same table | Foreign key referencing the same table |

## Important Interview Points

1. A foreign key references a candidate key, commonly the primary key, in the parent table.
2. A one-to-many foreign key belongs on the many side.
3. A many-to-many relationship needs a junction table in a relational design.
4. A `UNIQUE` foreign key can enforce one-to-one cardinality.
5. `NOT NULL` makes a foreign-key reference mandatory for each child row.
6. `ON DELETE CASCADE` can delete dependent child rows; use it only when appropriate.
7. ER diagrams are used to model entities, attributes, relationships, cardinality, and optionality.

## Practice Checklist

- [ ] Create a one-to-one relationship using a UNIQUE foreign key.
- [ ] Create a one-to-many customer-order relationship.
- [ ] Create a many-to-many student-course relationship.
- [ ] Create a self-referencing employee-manager table.
- [ ] Explain parent and child tables.
- [ ] Demonstrate referential integrity.
- [ ] Compare RESTRICT, CASCADE, and SET NULL.
- [ ] Explain cardinality and optionality.
- [ ] Convert a simple ER design into MySQL tables.

**End of Part 10 — Topics 152–161.**